# Stage 2b — HAM10000: Pushing Accuracy on the Lesion-Disjoint Split

**Goal.** Raise test accuracy on the *same* lesion-disjoint split as Stage 2 (7,145 / 1,429 / 1,441),
without reintroducing leakage and without tuning on the test set.

**What changes vs Stage 2 (each is a standard, defensible lever):**

| Lever | Stage 2 | Stage 2b |
|---|---|---|
| Backbones | 11 ImageNet CNNs @224 | EfficientNetV2S, ConvNeXtTiny @384, EfficientNetB3 @300 |
| Fine-tuning | top 30% unfrozen, 12 epochs | whole backbone, up to 30 epochs, cosine LR + warm-up, AdamW |
| Loss | CE + fully balanced class weights | CE + label smoothing 0.1, class weights **off** by default |
| Augmentation | flips, rot90, colour jitter | + rotation, zoom, translation, hue |
| Inference | single pass | 4-way flip test-time augmentation |
| Final model | single backbone | soft-voting ensemble, members chosen on **validation** |

**Rules this notebook enforces**
1. The split is loaded from Stage 2 (or rebuilt identically) and re-checked for zero lesion overlap.
2. Every choice — checkpoint epoch, ensemble members, thresholds — is made on **validation only**.
3. Test metrics are printed **once**, in the final section, after all choices are frozen.

**Trade-off to know before you start.** Turning class weights off raises accuracy mainly by predicting
the majority class (`nv`, 67%) more often; melanoma recall usually falls. Report accuracy *and*
macro-F1 / balanced accuracy / melanoma recall, never accuracy alone.

**Runtime.** Roughly 2–4 h on Kaggle T4×2 for all three models. Each model's predictions are saved as
soon as it finishes; re-running skips finished models.

In [5]:
import os, json, gc, time, itertools, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, roc_auc_score)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ paths
DATA          = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
STAGE2_SPLITS = Path("/kaggle/input/notebooks/shafinbubt/stage-2/stage2/splits.csv")  # attach Stage 2 output if you can
OUT           = Path("/kaggle/working/stage2b")
OUT.mkdir(parents=True, exist_ok=True); (OUT / "figs").mkdir(exist_ok=True)

# ------------------------------------------------------------------ experiment
SEED        = 42
QUICK_RUN   = False            # True -> tiny subset, 1+1 epochs: smoke-test the pipeline first (~10 min)
WEIGHTS     = "imagenet"

# name -> (keras.applications constructor, input size, batch size PER GPU)
MODELS = {
    "EfficientNetV2S": ("EfficientNetV2S", 384, 16),
    "ConvNeXtTiny":    ("ConvNeXtTiny",    384, 16),
    "EfficientNetB3":  ("EfficientNetB3",  300, 16),
}

EPOCHS_HEAD   = 2        # frozen backbone: warm up the new head
EPOCHS_FT     = 30       # whole backbone unfrozen
LR_HEAD       = 1e-3
LR_FT         = 1e-4     # peak LR after 1 warm-up epoch, cosine decay to 0
WEIGHT_DECAY  = 1e-5
LABEL_SMOOTH  = 0.1
DROPOUT       = 0.3
PATIENCE      = 7        # early stopping on the validation selection metric

CLASS_WEIGHT_MODE = "none"      # "none" (max accuracy) | "sqrt" (compromise) | "balanced" (Stage 2 behaviour)
SELECT_METRIC     = "accuracy"  # validation metric for checkpoints & ensemble: "accuracy" | "macro_f1"
TTA               = True

if QUICK_RUN:
    EPOCHS_HEAD, EPOCHS_FT, PATIENCE = 1, 1, 1

keras.utils.set_random_seed(SEED)
keras.mixed_precision.set_global_policy("mixed_float16")
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
strategy = tf.distribute.MirroredStrategy()
print("TF", tf.__version__, "| replicas:", strategy.num_replicas_in_sync)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
TF 2.20.0 | replicas: 2


## 1. Load the Stage 2 split and re-verify it

Using the identical split is what makes the new numbers comparable to Table II of the paper.
If Stage 2's `splits.csv` is not attached, the split is rebuilt with the same code and seed.

In [6]:
meta = pd.read_csv(DATA / "HAM10000_metadata.csv")

img_paths = {}
for d in DATA.rglob("*"):
    if d.is_dir() and d.name.startswith("HAM10000_images"):
        for f in d.glob("*.jpg"):
            img_paths[f.stem] = str(f)
if not img_paths:
    for f in DATA.rglob("*.jpg"):
        img_paths[f.stem] = str(f)
meta["path"] = meta["image_id"].map(img_paths)
assert meta["path"].notna().all(), "images missing — check DATA path"

CLASSES = sorted(meta["dx"].unique()); NUM_CLASSES = len(CLASSES)
meta["y"] = meta["dx"].map({c: i for i, c in enumerate(CLASSES)})

if STAGE2_SPLITS.exists():
    s2 = pd.read_csv(STAGE2_SPLITS)[["image_id", "split"]]
    meta = meta.merge(s2, on="image_id", how="left", validate="one_to_one")
    source = f"loaded from {STAGE2_SPLITS}"
else:
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
    trval_idx, test_idx = next(sgkf.split(meta, meta["y"], groups=meta["lesion_id"]))
    sub = meta.iloc[trval_idx].reset_index(drop=True)
    sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
    tr_idx, val_idx = next(sgkf2.split(sub, sub["y"], groups=sub["lesion_id"]))
    meta["split"] = None
    meta.loc[meta.index[test_idx], "split"] = "test"
    meta.loc[meta.index[trval_idx[tr_idx]], "split"] = "train"
    meta.loc[meta.index[trval_idx[val_idx]], "split"] = "val"
    source = "rebuilt with the Stage 2 code and seed"

# INTEGRITY GATE
les = {s: set(meta.loc[meta["split"] == s, "lesion_id"]) for s in ["train", "val", "test"]}
assert meta["split"].notna().all(), "unassigned images"
assert not (les["train"] & les["test"] or les["train"] & les["val"] or les["val"] & les["test"]), \
    "LESION LEAKAGE — do not train"
counts = meta["split"].value_counts()
print(f"split {source}")
print(f"train {counts['train']} | val {counts['val']} | test {counts['test']}   (Stage 2: 7145 | 1429 | 1441)")
print("INTEGRITY GATE: PASS — zero lesion overlap\n")
print(meta.groupby(["split", "dx"]).size().unstack(fill_value=0).loc[["train", "val", "test"]])

Y     = meta["y"].to_numpy().astype(np.int32)
PATHS = meta["path"].to_numpy()
IDX   = {s: np.where(meta["split"] == s)[0] for s in ["train", "val", "test"]}
if QUICK_RUN:
    rng = np.random.default_rng(SEED)
    IDX = {s: np.sort(rng.choice(v, size=min(len(v), 320), replace=False)) for s, v in IDX.items()}
Y_VAL, Y_TEST = Y[IDX["val"]], Y[IDX["test"]]
maj = np.bincount(Y[IDX["train"]]).argmax()
print(f"\nmajority-class baseline on TEST: acc {np.mean(Y_TEST == maj):.4f}")

split rebuilt with the Stage 2 code and seed
train 7145 | val 1429 | test 1441   (Stage 2: 7145 | 1429 | 1441)
INTEGRITY GATE: PASS — zero lesion overlap

dx     akiec  bcc  bkl  df  mel    nv  vasc
split                                      
train    244  358  780  79  814  4769   101
val       49   67  151  27  153   962    20
test      34   89  168   9  146   974    21

majority-class baseline on TEST: acc 0.6759


## 2. Input pipeline

Images are decoded and resized once, cached in RAM as `uint8`, then augmented per batch on the
fly — augmentation never touches validation or test. All three backbones carry their own input
rescaling inside the model, so they receive raw 0–255 pixels.

In [7]:
AUTOTUNE = tf.data.AUTOTUNE
L = keras.layers

def load(path, size):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (size, size), antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0.0, 255.0), tf.uint8)

AUG = keras.Sequential([
    L.RandomFlip("horizontal_and_vertical", dtype="float32"),
    L.RandomRotation(0.25, fill_mode="reflect", dtype="float32"),
    L.RandomZoom((-0.25, 0.10), fill_mode="reflect", dtype="float32"),
    L.RandomTranslation(0.10, 0.10, fill_mode="reflect", dtype="float32"),
    L.RandomBrightness(0.15, value_range=(0.0, 255.0), dtype="float32"),
    L.RandomContrast(0.20, dtype="float32"),
], name="augment")

def colour(x):
    x = tf.image.random_saturation(x / 255.0, 0.8, 1.2)
    x = tf.image.random_hue(x, 0.02)
    return tf.clip_by_value(x, 0.0, 1.0) * 255.0

def train_ds(size, batch):
    idx = IDX["train"]
    ds = tf.data.Dataset.from_tensor_slices((PATHS[idx], Y[idx]))
    ds = ds.map(lambda p, y: (load(p, size), y), num_parallel_calls=AUTOTUNE).cache()
    ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch, drop_remainder=True)
    ds = ds.map(lambda x, y: (colour(AUG(tf.cast(x, tf.float32), training=True)),
                              tf.one_hot(y, NUM_CLASSES)), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def eval_ds(split, size, batch=32):
    ds = tf.data.Dataset.from_tensor_slices(PATHS[IDX[split]])
    ds = ds.map(lambda p: load(p, size), num_parallel_calls=AUTOTUNE).cache()
    return ds.batch(batch).map(lambda x: tf.cast(x, tf.float32)).prefetch(AUTOTUNE)

FLIPS = [lambda x: x,
         tf.image.flip_left_right,
         tf.image.flip_up_down,
         lambda x: tf.image.flip_up_down(tf.image.flip_left_right(x))]

def predict(model, ds, tta):
    views = FLIPS if tta else FLIPS[:1]
    return np.mean([model.predict(ds.map(v), verbose=0) for v in views], axis=0)

def class_weights():
    if CLASS_WEIGHT_MODE == "none":
        return None
    n = np.bincount(Y[IDX["train"]], minlength=NUM_CLASSES).astype(float)
    w = n.sum() / (NUM_CLASSES * n)
    if CLASS_WEIGHT_MODE == "sqrt":
        w = np.sqrt(w)
    return {i: float(v) for i, v in enumerate(w)}

## 3. Model and training loop

- The backbone is called with `training=False`, so batch-norm statistics stay frozen even after
  the weights are unfrozen — the standard Keras recipe for fine-tuning EfficientNet on small data.
- Phase 1 trains only the head; phase 2 unfreezes the **whole** backbone with a 1-epoch warm-up
  and cosine decay.
- A callback scores **validation** after every epoch, keeps the best weights by `SELECT_METRIC`,
  and stops after `PATIENCE` epochs without improvement.

In [8]:
A = keras.applications
CTORS = {"EfficientNetV2S": A.EfficientNetV2S, "ConvNeXtTiny": A.ConvNeXtTiny,
         "EfficientNetB3": A.EfficientNetB3, "EfficientNetB4": A.EfficientNetB4,
         "ConvNeXtSmall": A.ConvNeXtSmall, "DenseNet201": A.DenseNet201}

def build(ctor, size):
    base = CTORS[ctor](weights=WEIGHTS, include_top=False,
                       input_shape=(size, size, 3), pooling="avg")
    inp = keras.Input((size, size, 3))
    x = base(inp, training=False)
    x = L.Dropout(DROPOUT)(x)
    out = L.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out), base

class ValSelector(keras.callbacks.Callback):
    # Scores validation each epoch, keeps the best weights, early-stops. Never sees test.
    def __init__(self, ds, y, path, patience):
        super().__init__()
        self.ds, self.y, self.path, self.patience = ds, y, str(path), patience
        self.best, self.wait, self.history = -1.0, 0, []
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.ds, verbose=0).argmax(1)
        acc = accuracy_score(self.y, p)
        f1 = f1_score(self.y, p, average="macro", zero_division=0)
        self.history.append({"acc": acc, "macro_f1": f1})
        score = acc if SELECT_METRIC == "accuracy" else f1
        flag = ""
        if score > self.best + 1e-4:
            self.best, self.wait, flag = score, 0, "  <- best"
            self.model.save_weights(self.path)
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                flag = "  (early stop)"
        print(f"    val acc {acc:.4f} | val macro-F1 {f1:.4f}{flag}")

def train_one(name):
    ctor, size, per_gpu = MODELS[name]
    batch = per_gpu * strategy.num_replicas_in_sync
    tr, va = train_ds(size, batch), eval_ds("val", size)
    steps = len(IDX["train"]) // batch
    loss = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)
    ckpt = OUT / f"{name}.weights.h5"
    sel = ValSelector(va, Y_VAL, ckpt, PATIENCE)

    with strategy.scope():
        model, base = build(ctor, size)
        base.trainable = False
        model.compile(optimizer=keras.optimizers.AdamW(LR_HEAD, weight_decay=WEIGHT_DECAY),
                      loss=loss, metrics=["accuracy"])
    print(f"  phase 1: head only ({EPOCHS_HEAD} epochs)")
    model.fit(tr, epochs=EPOCHS_HEAD, class_weight=class_weights(), callbacks=[sel], verbose=2)

    with strategy.scope():
        base.trainable = True
        sched = keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=LR_FT / 100, decay_steps=steps * max(EPOCHS_FT - 1, 1),
            warmup_target=LR_FT, warmup_steps=steps)
        model.compile(optimizer=keras.optimizers.AdamW(sched, weight_decay=WEIGHT_DECAY),
                      loss=loss, metrics=["accuracy"])
    sel.wait = 0
    print(f"  phase 2: full backbone (up to {EPOCHS_FT} epochs)")
    model.fit(tr, epochs=EPOCHS_FT, class_weight=class_weights(), callbacks=[sel], verbose=2)

    model.load_weights(str(ckpt))
    p_val  = predict(model, va, TTA)
    p_test = predict(model, eval_ds("test", size), TTA)   # saved now, only SCORED in section 6
    np.save(OUT / f"val_{name}.npy", p_val)
    np.save(OUT / f"test_{name}.npy", p_test)
    json.dump(sel.history, open(OUT / f"history_{name}.json", "w"))
    del model, base, tr, va; keras.backend.clear_session(); gc.collect()
    return p_val

## 4. Train

Resumable: a model whose `val_*.npy` and `test_*.npy` already exist in the output folder is skipped.
Only validation numbers are printed here.

In [9]:
for name in MODELS:
    if (OUT / f"val_{name}.npy").exists() and (OUT / f"test_{name}.npy").exists():
        print(f"skip {name} (already trained)"); continue
    print(f"\n{'=' * 70}\n{name}  @{MODELS[name][1]}px\n{'=' * 70}")
    t = time.time()
    p = train_one(name)
    print(f"-> {name}: VAL acc {accuracy_score(Y_VAL, p.argmax(1)):.4f} | "
          f"VAL macro-F1 {f1_score(Y_VAL, p.argmax(1), average='macro'):.4f} "
          f"(TTA={TTA}) | {(time.time() - t) / 60:.1f} min")


EfficientNetV2S  @384px
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  phase 1: head only (2 epochs)
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/t

I0000 00:00:1790676962.884261     129 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-09-29 10:16:07.811647: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-29 10:16:07.877847: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-29 10:16:07.951850: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-29 10:16:08.021598: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-o

    val acc 0.6984 | val macro-F1 0.2633  <- best
223/223 - 225s - 1s/step - accuracy: 0.6427 - loss: 1.3126
Epoch 2/2
    val acc 0.7306 | val macro-F1 0.3833  <- best
223/223 - 200s - 896ms/step - accuracy: 0.7068 - loss: 1.0855
  phase 2: full backbone (up to 30 epochs)
Epoch 1/30
INFO:tensorflow:Collective all_reduce tensors: 182 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


2026-09-29 10:24:01.567762: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-29 10:24:01.708983: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-29 10:24:01.720026: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-29 10:24:01.860673: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


    val acc 0.7460 | val macro-F1 0.3954  <- best
223/223 - 467s - 2s/step - accuracy: 0.7447 - loss: 0.9862
Epoch 2/30
    val acc 0.8027 | val macro-F1 0.6562  <- best
223/223 - 398s - 2s/step - accuracy: 0.8027 - loss: 0.8707
Epoch 3/30
    val acc 0.8307 | val macro-F1 0.6922  <- best
223/223 - 397s - 2s/step - accuracy: 0.8454 - loss: 0.7779
Epoch 4/30
    val acc 0.8111 | val macro-F1 0.7101
223/223 - 398s - 2s/step - accuracy: 0.8688 - loss: 0.7344
Epoch 5/30
    val acc 0.8425 | val macro-F1 0.7126  <- best
223/223 - 400s - 2s/step - accuracy: 0.8964 - loss: 0.6814
Epoch 6/30
    val acc 0.8726 | val macro-F1 0.7944  <- best
223/223 - 399s - 2s/step - accuracy: 0.9092 - loss: 0.6539
Epoch 7/30
    val acc 0.7985 | val macro-F1 0.7088
223/223 - 394s - 2s/step - accuracy: 0.9292 - loss: 0.6111
Epoch 8/30
    val acc 0.8265 | val macro-F1 0.7238
223/223 - 395s - 2s/step - accuracy: 0.9430 - loss: 0.5875
Epoch 9/30
    val acc 0.8369 | val macro-F1 0.7398
223/223 - 397s - 2s/step -

...

-> ConvNeXtTiny: VAL acc 0.8698 | VAL macro-F1 0.7862 (TTA=True) | 98.0 min

EfficientNetB3  @300px
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  phase 1: head only (2 epochs)
Epoch 1/2
INFO:tensorflow:Collective all_reduce tensors: 2 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


E0000 00:00:1790682838.063178      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/efficientnetb3_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


    val acc 0.7152 | val macro-F1 0.2603  <- best
223/223 - 139s - 622ms/step - accuracy: 0.6781 - loss: 1.1581
Epoch 2/2
    val acc 0.7306 | val macro-F1 0.3023  <- best
223/223 - 109s - 488ms/step - accuracy: 0.7056 - loss: 1.0677
  phase 2: full backbone (up to 30 epochs)
Epoch 1/30
INFO:tensorflow:Collective all_reduce tensors: 340 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


E0000 00:00:1790683157.499242      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/efficientnetb3_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


    val acc 0.7215 | val macro-F1 0.2963
223/223 - 298s - 1s/step - accuracy: 0.5737 - loss: 1.3697
Epoch 2/30
    val acc 0.7915 | val macro-F1 0.5446  <- best
223/223 - 187s - 838ms/step - accuracy: 0.7677 - loss: 0.9406
Epoch 3/30
    val acc 0.8013 | val macro-F1 0.5712  <- best
223/223 - 187s - 838ms/step - accuracy: 0.8023 - loss: 0.8674
Epoch 4/30
    val acc 0.8048 | val macro-F1 0.6068  <- best
223/223 - 186s - 833ms/step - accuracy: 0.8254 - loss: 0.8242
Epoch 5/30
    val acc 0.8020 | val macro-F1 0.6004
223/223 - 186s - 832ms/step - accuracy: 0.8450 - loss: 0.7835
Epoch 6/30
    val acc 0.8062 | val macro-F1 0.6111  <- best
223/223 - 184s - 824ms/step - accuracy: 0.8610 - loss: 0.7510
Epoch 7/30
    val acc 0.8174 | val macro-F1 0.6424  <- best
223/223 - 186s - 835ms/step - accuracy: 0.8761 - loss: 0.7284
Epoch 8/30
    val acc 0.8055 | val macro-F1 0.6624
223/223 - 181s - 809ms/step - accuracy: 0.8835 - loss: 0.7008
Epoch 9/30
    val acc 0.8230 | val macro-F1 0.6914  <- b

## 5. Choose the final configuration — on validation only

Every non-empty subset of the trained models is scored as an equal-weight soft-voting ensemble
on validation; the best one by `SELECT_METRIC` is frozen. Separately, per-class decision
thresholds are fitted on validation to maximise macro-F1 (the Stage 3 method), giving a second,
clearly labelled operating point. Nothing in this section reads the test labels.

In [10]:
names = [n for n in MODELS if (OUT / f"val_{n}.npy").exists()]
PV = {n: np.load(OUT / f"val_{n}.npy") for n in names}

def score(y, p):
    pred = p.argmax(1)
    return {"acc": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0)}

rows = []
for k in range(1, len(names) + 1):
    for combo in itertools.combinations(names, k):
        s = score(Y_VAL, np.mean([PV[n] for n in combo], 0))
        rows.append({"members": " + ".join(combo), **s})
sel_df = pd.DataFrame(rows).sort_values("acc" if SELECT_METRIC == "accuracy" else "macro_f1",
                                        ascending=False)
print("VALIDATION — all ensembles\n")
print(sel_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
CHOSEN = sel_df.iloc[0]["members"].split(" + ")
print(f"\nCHOSEN (by val {SELECT_METRIC}): {CHOSEN}")

# per-class threshold calibration for macro-F1, fitted on validation only
pv = np.mean([PV[n] for n in CHOSEN], 0)
t = np.ones(NUM_CLASSES)
best = f1_score(Y_VAL, (pv / t).argmax(1), average="macro", zero_division=0)
for _ in range(6):
    improved = False
    for c in range(NUM_CLASSES):
        for cand in np.geomspace(0.2, 5.0, 25):
            t2 = t.copy(); t2[c] = cand
            s = f1_score(Y_VAL, (pv / t2).argmax(1), average="macro", zero_division=0)
            if s > best + 1e-5:
                best, t, improved = s, t2, True
    if not improved:
        break
print(f"val macro-F1 with calibrated thresholds: {best:.4f}")

FROZEN = {"chosen_members": CHOSEN, "select_metric": SELECT_METRIC, "tta": TTA,
          "class_weight_mode": CLASS_WEIGHT_MODE, "thresholds": t.tolist(),
          "models": {n: MODELS[n] for n in names}}
json.dump(FROZEN, open(OUT / "frozen_choices.json", "w"), indent=2)
print("choices frozen ->", OUT / "frozen_choices.json")

VALIDATION — all ensembles

                                        members    acc  macro_f1
                                   ConvNeXtTiny 0.8698    0.7862
EfficientNetV2S + ConvNeXtTiny + EfficientNetB3 0.8684    0.7908
                 EfficientNetV2S + ConvNeXtTiny 0.8684    0.7784
                  ConvNeXtTiny + EfficientNetB3 0.8670    0.7840
               EfficientNetV2S + EfficientNetB3 0.8579    0.7611
                                EfficientNetV2S 0.8481    0.7343
                                 EfficientNetB3 0.8453    0.7584

CHOSEN (by val accuracy): ['ConvNeXtTiny']
val macro-F1 with calibrated thresholds: 0.8135
choices frozen -> /kaggle/working/stage2b/frozen_choices.json


## 6. Final test evaluation — run once

Everything below uses the choices frozen in section 5. Do not go back and change settings because
of what this cell prints; if you do, the test set has become a validation set and the numbers
are no longer an honest estimate.

In [11]:
FROZEN = json.load(open(OUT / "frozen_choices.json"))
PT = {n: np.load(OUT / f"test_{n}.npy") for n in FROZEN["models"]}
thr = np.array(FROZEN["thresholds"])

def full(y, p, t=None):
    pred = (p / t).argmax(1) if t is not None else p.argmax(1)
    try:
        auc = roc_auc_score(y, p, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")
    return {"acc": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "bal_acc": balanced_accuracy_score(y, pred), "macro_auc": auc,
            "mel_recall": float(np.mean(pred[y == CLASSES.index("mel")] == CLASSES.index("mel")))}, pred

def boot_ci(y, pred, n=2000):
    r = np.random.default_rng(SEED)
    accs = [np.mean(y[i] == pred[i]) for i in (r.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(accs, [2.5, 97.5])

rows = []
for n, p in PT.items():
    m, _ = full(Y_TEST, p); rows.append({"config": f"{n} (TTA={FROZEN['tta']})", **m})
p_ens = np.mean([PT[n] for n in FROZEN["chosen_members"]], 0)
m_ens, pred_ens = full(Y_TEST, p_ens)
rows.append({"config": "ENSEMBLE: " + " + ".join(FROZEN["chosen_members"]), **m_ens})
m_cal, pred_cal = full(Y_TEST, p_ens, thr)
rows.append({"config": "ENSEMBLE + val-calibrated thresholds", **m_cal})
res = pd.DataFrame(rows)
print("TEST — lesion-disjoint HAM10000\n")
print(res.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
lo, hi = boot_ci(Y_TEST, pred_ens)
print(f"\nensemble accuracy {m_ens['acc']:.4f}   95% bootstrap CI [{lo:.4f}, {hi:.4f}]")
print(f"majority-class baseline accuracy {np.mean(Y_TEST == maj):.4f}")
print(f"Stage 2 reference (ResNet50@224): acc 0.7280 | macro-F1 0.5971")
res.to_csv(OUT / "test_results.csv", index=False)

print("\nper-class report — ensemble (argmax)")
print(classification_report(Y_TEST, pred_ens, target_names=CLASSES, digits=4, zero_division=0))

cm = confusion_matrix(Y_TEST, pred_ens, normalize="true")
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(NUM_CLASSES), CLASSES, rotation=45); plt.yticks(range(NUM_CLASSES), CLASSES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                 color="white" if cm[i, j] > .5 else "black", fontsize=8)
plt.title("Ensemble — row-normalised confusion matrix (test)"); plt.colorbar(); plt.tight_layout()
plt.savefig(OUT / "figs" / "confusion_ensemble.png", dpi=200)

TEST — lesion-disjoint HAM10000

                              config    acc  macro_f1  bal_acc  macro_auc  mel_recall
          EfficientNetV2S (TTA=True) 0.8508    0.7051   0.7158     0.9658      0.4726
             ConvNeXtTiny (TTA=True) 0.8605    0.7386   0.7320     0.9723      0.6644
           EfficientNetB3 (TTA=True) 0.8369    0.6701   0.6443     0.9575      0.4795
              ENSEMBLE: ConvNeXtTiny 0.8605    0.7386   0.7320     0.9723      0.6644
ENSEMBLE + val-calibrated thresholds 0.8452    0.7245   0.7200     0.9723      0.6781

ensemble accuracy 0.8605   95% bootstrap CI [0.8425, 0.8786]
majority-class baseline accuracy 0.6759
Stage 2 reference (ResNet50@224): acc 0.7280 | macro-F1 0.5971

per-class report — ensemble (argmax)
              precision    recall  f1-score   support

       akiec     0.6800    0.5000    0.5763        34
         bcc     0.8675    0.8090    0.8372        89
         bkl     0.7143    0.7440    0.7289       168
          df     0.6667    0.66

## Reading the results

- **Compare like with like.** Same split as Stage 2, so the gain over 72.80% is real — but it comes
  from a bigger input size, stronger backbones, full fine-tuning, TTA and ensembling together.
  Say so if you report it; don't present it as the Stage 2 harness.
- **Check what the accuracy is made of.** If accuracy rose but melanoma recall and balanced
  accuracy fell, the model is leaning on `nv`. Re-run with `CLASS_WEIGHT_MODE = "sqrt"` and
  `SELECT_METRIC = "macro_f1"` for the clinically more meaningful operating point, and report both.
- **One split, one seed.** The bootstrap CI covers test-sample noise, not training noise. For a
  claim like "above 85%", repeat with 2–3 seeds and report mean ± std.
- **If you are still short of 85%.** Next levers, in order: add `EfficientNetB4` @380 to `MODELS`;
  raise `EPOCHS_FT` to 40; move to 5-fold lesion-grouped cross-validation and average the folds.
  Change these based on *validation* results only.